In [2]:
import json
import re
import random
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.naive_bayes import ComplementNB

In [3]:
URL_RE     = re.compile(r"(https?://\S+|www\.\S+|\{URL\})", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")

In [5]:
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = URL_RE.sub(" [URL] ", s)
    s = MENTION_RE.sub(" [MENTION] ", s)
    s = HASHTAG_RE.sub(r" [HASHTAG:\1] ", s)
    s = re.sub(r"\s+", " ", s)
    return s

def eval_split(model, X, y, split_name="VAL"):
    """
    Evalúa el modelo y devuelve un diccionario con métricas clave.
    También imprime el informe detallado.
    """
    preds = model.predict(X)

    f1m = f1_score(y, preds, average="macro")
    acc = accuracy_score(y, preds)

    print(f"\n=== {split_name} ===")
    print(classification_report(y, preds, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y, preds))
    print("F1-macro:", f"{f1m:.4f}")
    print("Accuracy:", f"{acc:.4f}")

    return {
        "f1_macro": f1m,
        "accuracy": acc,
    }

def eval_split(model, X, y, split_name="VAL"):
    """
    Evalúa el modelo y devuelve un diccionario con métricas clave.
    También imprime el informe detallado.
    """
    preds = model.predict(X)

    f1m = f1_score(y, preds, average="macro")
    acc = accuracy_score(y, preds)

    print(f"\n=== {split_name} ===")
    print(classification_report(y, preds, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y, preds))
    print("F1-macro:", f"{f1m:.4f}")
    print("Accuracy:", f"{acc:.4f}")

    return {
        "f1_macro": f1m,
        "accuracy": acc,
    }

In [6]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

def load_ethos(path):
    df = pd.read_csv(path)

    # Drop rows without tweet text
    df = df[df["tweet"].astype(str).str.strip() != ""].copy()

    # Clean text
    df["clean"] = df["tweet"].apply(clean_text)

    # Binary hate label (adapt if your coding is different)
    df["label"] = (df["sentiment"].astype(str).str.lower() == "hate").astype(int)

    # Version with only tweet and target
    df_min = df[["tweet", "target"]].copy()

    return df, df_min

In [7]:
def load_and_clean_tweethate(path, text_field="text"):
    """
    Carga un JSONL tipo supertweeteval, filtra texto vacío,
    elimina NaN reales y NaN en string en predicted_hate_category,
    genera label binaria y limpia el texto.

    Devuelve:
        df : DataFrame limpio y con columnas:
             - text_field
             - predicted_hate_category
             - label (0/1)
             - clean (texto limpio)
    """
    # ====== Carga ======
    rows = load_jsonl(path)
    df = pd.DataFrame(rows)

    # ====== 1. Filtrar texto no vacío ======
    df = df[df[text_field].apply(lambda x: isinstance(x, str) and x.strip() != "")].copy()

    # ====== 2. Eliminar NaN reales ======
    df = df.dropna(subset=["predicted_hate_category"]).copy()

    # ====== 3. Eliminar "nan" como texto ======
    df = df[df["predicted_hate_category"].astype(str).str.lower() != "nan"].copy()

    # ====== 4. Etiqueta binaria ======
    df["label"] = (
        df["predicted_hate_category"]
        .astype(str)
        .str.lower()
        != "nan"
    ).astype(int)

    # ====== 5. Texto limpio ======
    df["clean"] = df[text_field].apply(clean_text)

    return df

# --------------------
# 2) Load NEW CSV dataset
# --------------------
INPUT_CSV = "ethos/en_dataset_with_stop_words.csv"
df_new_full, df_new_min = load_ethos(INPUT_CSV)

In [8]:
INPUT_JSONL = "supertweeteval/merged_masked_unionv2.jsonl"

df_tweethate = load_and_clean_tweethate(INPUT_JSONL, text_field="text")
df_tweethate.head()


,id,text,text_masked,predicted_hate_category,groups_from_mask,taxonomy_version,masking_model,lang,lexicon,label,clean
2,10,If yall faggots waste votes on this clownery i...,If yall [SLUR:SEXUAL_ORIENTATION_OTHER] waste ...,hate_sexuality,[SEXUAL_ORIENTATION_OTHER],v1.1-closed,gpt-4o-mini,en,1,1,If yall faggots waste votes on this clownery i...
4,1000,These niggas are goofy ass scary hoes foreal lol,These [SLUR:RACE_BLACK] are goofy ass scary [S...,hate_race,"[GENDER_WOMEN, RACE_BLACK]",v1.1-closed,gpt-4o-mini,en,1,1,These niggas are goofy ass scary hoes foreal lol
8,1004,: females are literal reptiles : like born sna...,: [SLUR:GENDER_WOMEN] are literal reptiles : l...,hate_gender,[GENDER_WOMEN],v1.1-closed,gpt-4o-mini,en,1,1,: females are literal reptiles : like born sna...
14,101,@user worst thing about conservative women: wi...,[MENTION] worst thing about conservative [TARG...,hate_gender,[GENDER_WOMEN],v1.1-closed,gpt-4o-mini,en,1,1,[MENTION] worst thing about conservative wome...
15,1010,Women really be minding their business and tra...,[GENDER_WOMEN] really be minding their busines...,hate_gender,[GENDER_MEN],v1.1-closed,gpt-4o-mini,en,1,1,Women really be minding their business and tra...


In [9]:
def show_hate_and_target_distribution(df, name, label_col="label"):
    print(f"\n===== {name} – target distribution =====")
    print(df[label_col].value_counts().rename("count"))
    print("\nProportions:")
    print(df[label_col].value_counts(normalize=True).round(3).rename("prop"))

# Compare distributions
show_hate_and_target_distribution(df_new_full, "new CSV dataset", label_col="target")
show_hate_and_target_distribution(df_tweethate, "new CSV dataset", label_col="predicted_hate_category")



===== new CSV dataset – target distribution =====
target
origin                2448
disability            1089
other                  890
gender                 638
sexual_orientation     514
religion                68
Name: count, dtype: int64

Proportions:
target
origin                0.434
disability            0.193
other                 0.158
gender                0.113
sexual_orientation    0.091
religion              0.012
Name: prop, dtype: float64

===== new CSV dataset – target distribution =====
predicted_hate_category
hate_gender        771
hate_race          542
hate_sexuality     246
hate_religion      234
hate_origin        162
hate_disability     20
hate_age             9
Name: count, dtype: int64

Proportions:
predicted_hate_category
hate_gender        0.389
hate_race          0.273
hate_sexuality     0.124
hate_religion      0.118
hate_origin        0.082
hate_disability    0.010
hate_age           0.005
Name: prop, dtype: float64


In [ ]:
# Experiment function

In [10]:
# =========================================
# Helper: intervalo de confianza 95%
# =========================================
def ci95(series):
    arr = np.array(series)
    mean = arr.mean()
    std  = arr.std(ddof=1)
    ci_low  = mean - 1.96 * std / np.sqrt(len(arr))
    ci_high = mean + 1.96 * std / np.sqrt(len(arr))
    return mean, std, ci_low, ci_high


# =========================================
# Experimento por seed (tu versión limpia)
# =========================================
def _single_seed_experiment(df_trainval, df_test, X_field, seed,
                            val_size, ngram_range, min_df, max_df, max_features):

    random.seed(seed)
    np.random.seed(seed)

    df_trainval = df_trainval[df_trainval[X_field].apply(lambda x: isinstance(x, str) and x.strip() != "")]
    df_test     = df_test[df_test[X_field].apply(lambda x: isinstance(x, str) and x.strip() != "")]

    for df in (df_trainval, df_test):
        df["phc"] = df["predicted_hate_category"].astype(str).str.strip()

    df_trainval = df_trainval[df_trainval["phc"].str.lower() != "nan"].copy()
    df_test     = df_test[df_test["phc"].str.lower() != "nan"].copy()

    if len(df_trainval) == 0 or len(df_test) == 0:
        raise RuntimeError("Train/Val o Test vacío tras filtrar 'nan'.")

    # Etiquetado
    all_labels = sorted(set(df_trainval["phc"].str.lower()) | set(df_test["phc"].str.lower()))
    label2id = {c:i for i,c in enumerate(all_labels)}
    id2label = {i:c for c,i in label2id.items()}

    for df in (df_trainval, df_test):
        df["label"] = df["phc"].str.lower().map(label2id).astype(int)
        df["clean"] = df[X_field].apply(clean_text)

    # Split
    X_train, X_val, y_train, y_val = train_test_split(
        df_trainval["clean"].values,
        df_trainval["label"].values,
        test_size=val_size,
        random_state=seed,
        stratify=df_trainval["label"].values
    )
    X_test = df_test["clean"].values
    y_test = df_test["label"].values

    # Vectorizer
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        strip_accents="unicode",
        sublinear_tf=True,
        stop_words="english",
        max_features=max_features,
    )
    Xtr = vectorizer.fit_transform(X_train)
    Xva = vectorizer.transform(X_val)
    Xte = vectorizer.transform(X_test)

    # Pesos de clase
    uniq = np.unique(y_train)
    class_weights = compute_class_weight(class_weight="balanced", classes=uniq, y=y_train)
    cw_map = {int(c): float(w) for c,w in zip(uniq, class_weights)}
    sw_train = np.vectorize(cw_map.get)(y_train)

    # Modelos
    models = {
        "logreg_ovr": LogisticRegression(
            max_iter=8000, n_jobs=-1, class_weight=cw_map, C=2.0,
            solver="liblinear", multi_class="ovr", random_state=seed),
        "linear_svm": LinearSVC(class_weight=cw_map, random_state=seed),
        "sgd_log": SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=2000,
                                 tol=1e-3, random_state=seed),
        "passive_aggressive": PassiveAggressiveClassifier(C=0.5, max_iter=2000,
                                                          random_state=seed),
        "complement_nb": ComplementNB(alpha=0.5),
    }

    seed_results = []

    for name, clf in models.items():
        # Entrenamiento
        try:
            clf.fit(Xtr, y_train, sample_weight=sw_train)
        except TypeError:
            clf.fit(Xtr, y_train)

        # Evaluación
        val_metrics  = eval_split(clf, Xva, y_val, "VAL")
        test_metrics = eval_split(clf, Xte, y_test, "TEST (fijo)")

        seed_results.append({
            "model": name,
            "seed": seed,
            "val_f1_macro":  val_metrics["f1_macro"],
            "val_accuracy":  val_metrics["accuracy"],
            "test_f1_macro": test_metrics["f1_macro"],
            "test_accuracy": test_metrics["accuracy"],
        })

    return pd.DataFrame(seed_results)



# =========================================
# Experimento sobre múltiples seeds (main)
# =========================================
def experiment_from_df_multi_seed(
        df_trainval, df_test, X_field,
        seeds=[1,2,3,4,5],
        val_size=0.10, ngram_range=(1,2), 
        min_df=3, max_df=0.90, max_features=None):

    all_runs = []

    print("=====================================")
    print(" Ejecutando Seeds:", seeds)
    print("=====================================")

    for sd in seeds:
        print(f"\n##### SEED = {sd} #####\n")
        df_seed = _single_seed_experiment(
            df_trainval.copy(),
            df_test.copy(),
            X_field,
            seed=sd,
            val_size=val_size,
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            max_features=max_features
        )
        all_runs.append(df_seed)

    df_all = pd.concat(all_runs, ignore_index=True)

    # ============================
    #  Agregado por modelo
    # ============================
    rows = []
    for model, sub in df_all.groupby("model"):
        mean_val, std_val, lo_val, hi_val = ci95(sub["val_f1_macro"])
        mean_te, std_te, lo_te, hi_te = ci95(sub["test_f1_macro"])

        rows.append({
            "model": model,
            "val_f1_macro_mean": mean_val,
            "val_f1_macro_std": std_val,
            "val_f1_macro_CI_low": lo_val,
            "val_f1_macro_CI_high": hi_val,

            "test_f1_macro_mean": mean_te,
            "test_f1_macro_std": std_te,
            "test_f1_macro_CI_low": lo_te,
            "test_f1_macro_CI_high": hi_te,
        })

    df_summary = pd.DataFrame(rows).sort_values(by="val_f1_macro_mean", ascending=False)

    print("\n=====================================")
    print("Resumen Multi-Seed (95% CI)")
    print("=====================================")
    print(df_summary.to_string(index=False))

    return df_all, df_summary


In [11]:
def remove_values_from_column(df, column, values_to_remove):
    """
    Devuelve un nuevo DataFrame donde se eliminan todas las filas
    cuya columna `column` contiene uno de los valores en `values_to_remove`.

    Parámetros:
        df : DataFrame de entrada
        column : str, nombre de la columna a filtrar
        values_to_remove : lista, set o array con valores a eliminar

    Retorna:
        df_filtrado : DataFrame sin las filas indeseadas
    """
    # Convertimos a set para eficiencia
    remove_set = set(values_to_remove)

    # Filtrado
    df_filtered = df[~df[column].isin(remove_set)].copy()

    return df_filtered


In [12]:
def collapse_race_into_origin(df, col="predicted_hate_category"):
    """
    Reemplaza hate_race -> hate_origin en la columna dada.
    Devuelve un nuevo DataFrame (no in-place).
    """
    df = df.copy()
    df[col] = df[col].replace({"hate_race": "hate_origin"})
    return df


## Create Test set

## Initial test

In [12]:
#INPUT_JSONL = "supertweeteval/merged_masked_unionv2.jsonl"
TEST_JSONL = "supertweeteval/test_split_only_originals.jsonl"

INPUT_JSONL = "supertweeteval/broken_phone/balanced_with_broken_phone_supertweetevalv3.jsonl"
TEST_JSONL = "supertweeteval/test_split_only_originals.jsonl"

my_train_df = load_and_clean_tweethate(INPUT_JSONL, text_field="text")
my_test_df = load_and_clean_tweethate(TEST_JSONL, text_field="text")

my_train_df = remove_values_from_column(my_train_df, "predicted_hate_category", ["hate_age"])
my_test_df = remove_values_from_column(my_test_df, "predicted_hate_category", ["hate_age"])

my_train_df = collapse_race_into_origin(my_train_df, col="predicted_hate_category")
my_test_df      = collapse_race_into_origin(my_test_df,      col="predicted_hate_category")


In [ ]:
df_all_runs, df_summary = experiment_from_df_multi_seed(
    df_trainval=my_train_df,
    df_test=my_test_df,
    X_field="text",
    seeds=[7,11,13,17,19]
)

 Ejecutando Seeds: [7, 11, 13, 17, 19]

##### SEED = 7 #####


=== VAL ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        71
           1     0.8987    0.9467    0.9221        75
           2     0.9929    0.9459    0.9689       148
           3     0.9737    1.0000    0.9867        74
           4     0.9737    0.9867    0.9801        75

    accuracy                         0.9707       443
   macro avg     0.9678    0.9759    0.9715       443
weighted avg     0.9716    0.9707    0.9708       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  0  71   1   1   2]
 [  0   7 140   1   0]
 [  0   0   0  74   0]
 [  0   1   0   0  74]]
F1-macro: 0.9715
Accuracy: 0.9707

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000         1
           1     0.8750    0.8537    0.8642        41
           2     0.8529    0.7838    0.8169        37
           3     0.5882    0.

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1


=== VAL ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        71
           1     0.8256    0.9467    0.8820        75
           2     0.9776    0.8851    0.9291       148
           3     0.9487    1.0000    0.9737        74
           4     1.0000    0.9867    0.9933        75

    accuracy                         0.9503       443
   macro avg     0.9504    0.9637    0.9556       443
weighted avg     0.9544    0.9503    0.9508       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  0  71   3   1   0]
 [  0  14 131   3   0]
 [  0   0   0  74   0]
 [  0   1   0   0  74]]
F1-macro: 0.9556
Accuracy: 0.9503

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000         1
           1     0.8537    0.8537    0.8537        41
           2     0.8529    0.7838    0.8169        37
           3     0.5882    0.8333    0.6897        12
           4     1.0000    0.8462    

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1

              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000         1
           1     0.8333    0.8537    0.8434        41
           2     0.8235    0.7568    0.7887        37
           3     0.5625    0.7500    0.6429        12
           4     1.0000    0.8462    0.9167        13

    accuracy                         0.8077       104
   macro avg     0.8439    0.8413    0.8383       104
weighted avg     0.8210    0.8077    0.8115       104

Confusion matrix:
 [[ 1  0  0  0  0]
 [ 0 35  4  2  0]
 [ 0  4 28  5  0]
 [ 0  1  2  9  0]
 [ 0  2  0  0 11]]
F1-macro: 0.8383
Accuracy: 0.8077

=== VAL ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        71
           1     0.9600    0.9600    0.9600        75
           2     0.9863    0.9730    0.9796       148
           3     0.9737    1.0000    0.9867        74
           4     1.0000    1.0000    1.0000        75

    accuracy                

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"


=== VAL ===
              precision    recall  f1-score   support

           0     0.9861    1.0000    0.9930        71
           1     0.9706    0.8800    0.9231        75
           2     0.9603    0.9797    0.9699       148
           3     1.0000    1.0000    1.0000        74
           4     0.9615    1.0000    0.9804        75

    accuracy                         0.9729       443
   macro avg     0.9757    0.9719    0.9733       443
weighted avg     0.9730    0.9729    0.9725       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  1  66   6   0   2]
 [  0   2 145   0   1]
 [  0   0   0  74   0]
 [  0   0   0   0  75]]
F1-macro: 0.9733
Accuracy: 0.9729

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000         1
           1     0.8500    0.8293    0.8395        41
           2     0.8108    0.8108    0.8108        37
           3     0.6000    0.7500    0.6667        12
           4     0.9091    0.7692    

In [72]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.704571,0.045734,0.664483,0.744658,0.757229,0.013061,0.745780,0.768677
1,linear_svm,0.691243,0.049450,0.647898,0.734588,0.747702,0.012699,0.736571,0.758834
3,passive_aggressive,0.689503,0.058110,0.638567,0.740439,0.788131,0.031646,0.760392,0.815870
2,logreg_ovr,0.621616,0.023852,0.600708,0.642524,0.676250,0.021434,0.657463,0.695038
0,complement_nb,0.584789,0.029115,0.559268,0.610309,0.637153,0.009632,0.628710,0.645596


In [116]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
3,passive_aggressive,0.986146,0.004920,0.981834,0.990459,0.711398,0.116836,0.608986,0.813809
1,linear_svm,0.982166,0.003895,0.978752,0.985580,0.838836,0.012420,0.827949,0.849723
4,sgd_log,0.975202,0.004870,0.970934,0.979471,0.844118,0.015158,0.830831,0.857404
2,logreg_ovr,0.964467,0.006119,0.959103,0.969830,0.840554,0.012217,0.829846,0.851263
0,complement_nb,0.931495,0.014556,0.918736,0.944254,0.647087,0.009378,0.638867,0.655308


In [13]:
#INPUT_JSONL = "supertweeteval/merged_masked_unionv2.jsonl"
TEST_JSONL = "supertweeteval/test_split_only_originals.jsonl"
INPUT_JSONL = "supertweeteval/broken_phone/balanced_with_broken_phone_supertweetevalv3.jsonl"


my_train_df = load_and_clean_tweethate(INPUT_JSONL, text_field="text_masked")
my_test_df = load_and_clean_tweethate(TEST_JSONL, text_field="text_masked")

my_train_df = remove_values_from_column(my_train_df, "predicted_hate_category", ["hate_age"])
my_test_df = remove_values_from_column(my_test_df, "predicted_hate_category", ["hate_age"])

my_train_df = collapse_race_into_origin(my_train_df, col="predicted_hate_category")
my_test_df      = collapse_race_into_origin(my_test_df,      col="predicted_hate_category")

In [121]:
df_all_runs, df_summary = experiment_from_df_multi_seed(
    df_trainval=my_train_df,
    df_test=my_test_df,
    X_field="text_masked",
    seeds=[7,11,13,17,19]
)

 Ejecutando Seeds: [7, 11, 13, 17, 19]

##### SEED = 7 #####


=== VAL ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        71
           1     0.8588    0.9733    0.9125        75
           2     0.9854    0.9122    0.9474       148
           3     0.9324    0.9324    0.9324        74
           4     0.9605    0.9733    0.9669        75

    accuracy                         0.9503       443
   macro avg     0.9474    0.9583    0.9518       443
weighted avg     0.9533    0.9503    0.9507       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  0  73   0   0   2]
 [  0   8 135   5   0]
 [  0   2   2  69   1]
 [  0   2   0   0  73]]
F1-macro: 0.9518
Accuracy: 0.9503

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     0.5000    1.0000    0.6667         1
           1     0.8444    0.9268    0.8837        41
           2     0.8889    0.8649    0.8767        37
           3     1.0000    0.

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1


=== VAL ===
              precision    recall  f1-score   support

           0     0.9726    1.0000    0.9861        71
           1     0.7753    0.9200    0.8415        75
           2     0.9843    0.8446    0.9091       148
           3     0.8987    0.9595    0.9281        74
           4     0.9467    0.9467    0.9467        75

    accuracy                         0.9187       443
   macro avg     0.9155    0.9341    0.9223       443
weighted avg     0.9264    0.9187    0.9195       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  0  69   2   0   4]
 [  2  13 125   8   0]
 [  0   3   0  71   0]
 [  0   4   0   0  71]]
F1-macro: 0.9223
Accuracy: 0.9187

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     0.5000    1.0000    0.6667         1
           1     0.8444    0.9268    0.8837        41
           2     0.9167    0.8919    0.9041        37
           3     1.0000    0.5833    0.7368        12
           4     0.8571    0.9231    

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1

              precision    recall  f1-score   support

           0     0.5000    1.0000    0.6667         1
           1     0.8478    0.9512    0.8966        41
           2     0.8889    0.8649    0.8767        37
           3     1.0000    0.5833    0.7368        12
           4     0.9231    0.9231    0.9231        13

    accuracy                         0.8750       104
   macro avg     0.8320    0.8645    0.8200       104
weighted avg     0.8861    0.8750    0.8722       104

Confusion matrix:
 [[ 1  0  0  0  0]
 [ 0 39  2  0  0]
 [ 0  5 32  0  0]
 [ 1  1  2  7  1]
 [ 0  1  0  0 12]]
F1-macro: 0.8200
Accuracy: 0.8750

=== VAL ===
              precision    recall  f1-score   support

           0     0.9859    0.9859    0.9859        71
           1     0.8690    0.9733    0.9182        75
           2     0.9718    0.9324    0.9517       148
           3     0.9863    0.9730    0.9796        74
           4     0.9863    0.9600    0.9730        75

    accuracy                

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(


In [119]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.812323,0.037968,0.779043,0.845604,0.777282,0.013976,0.765032,0.789532
3,passive_aggressive,0.807570,0.040248,0.772291,0.842848,0.766606,0.034440,0.736418,0.796793
1,linear_svm,0.792903,0.050220,0.748883,0.836923,0.760166,0.010335,0.751107,0.769225
2,logreg_ovr,0.760215,0.017529,0.744850,0.775580,0.737553,0.008155,0.730404,0.744701
0,complement_nb,0.703088,0.014002,0.690815,0.715362,0.698448,0.023736,0.677643,0.719254


In [122]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
3,passive_aggressive,0.952840,0.012172,0.942171,0.963509,0.816670,0.002303,0.814651,0.818689
1,linear_svm,0.952786,0.014078,0.940446,0.965126,0.816422,0.006416,0.810798,0.822046
4,sgd_log,0.937986,0.012473,0.927053,0.948920,0.818397,0.006474,0.812723,0.824072
2,logreg_ovr,0.930899,0.016190,0.916709,0.945090,0.812485,0.005657,0.807526,0.817444
0,complement_nb,0.912296,0.012705,0.901160,0.923432,0.783456,0.023796,0.762598,0.804314


## Create new test

In [13]:
def map_new_csv_to_hate_categories(df_new_full, text_col="tweet"):
    """
    A partir de df_new_full con columnas [text_col, target],
    crea una columna predicted_hate_category compatible con df_tweethate.

    target:
        origin              -> hate_origin
        disability          -> hate_disability
        gender              -> hate_gender
        sexual_orientation  -> hate_sexuality
        religion            -> hate_religion
        other               -> se descarta (no mapeamos)

    Devuelve:
        df_mapped con columnas:
            - text
            - target
            - predicted_hate_category
            - clean
    """
    df = df_new_full.copy()

    # Mapeo target -> categoría hate_*
    target2hate = {
        "origin":             "hate_origin",
        "disability":         "hate_disability",
        "gender":             "hate_gender",
        "sexual_orientation": "hate_sexuality",
        "religion":           "hate_religion",
        # "other": None  # lo descartamos
    }

    df["predicted_hate_category"] = df["target"].map(target2hate)

    # Nos quedamos solo con targets que mapean a una categoría conocida
    df = df[~df["predicted_hate_category"].isna()].copy()

    # Normalizamos nombre de columna de texto para que coincida con df_tweethate (text)
    df["text"] = df[text_col]

    # Limpiamos texto igual que en el resto del pipeline
    df["clean"] = df["text"].apply(clean_text)

    return df


In [14]:
df_new_mapped = map_new_csv_to_hate_categories(df_new_full, text_col="tweet")

In [19]:
import numpy as np
import pandas as pd

def augment_test_with_new_data(
    df_test,
    df_new_mapped,
    label_col="predicted_hate_category",
    total_new=500,
    random_state=42,
    mode="proportional",
):
    """
    Añade ejemplos de df_new_mapped al df_test.

    Parámetros:
        df_test       : DataFrame de test original
        df_new_mapped : DataFrame con columnas [text, clean, label_col]
        label_col     : nombre de columna de clase (por defecto 'predicted_hate_category')
        total_new     : nº total de ejemplos nuevos a intentar añadir
        random_state  : semilla para reproducibilidad
        mode          : 
            - "proportional": mantiene aprox. la distribución de clases de df_test (modo original).
            - "balanced": reparte los nuevos ejemplos de forma aproximadamente uniforme
                          entre las clases presentes en df_test y df_new_mapped.

    Devuelve:
        df_test_aug   : test original + nuevos ejemplos
        df_added      : solo los ejemplos añadidos

    Nota:
        En modo "balanced" el nº real de ejemplos añadidos puede ser < total_new
        si alguna clase no tiene suficientes ejemplos en df_new_mapped.
    """
    df_test = df_test.copy()
    df_new  = df_new_mapped.copy()

    # Clases presentes en test (en orden de frecuencia)
    test_counts = df_test[label_col].value_counts()
    labels_in_test = list(test_counts.index)

    # Filtrar a clases que también existen en df_new
    labels_new = set(df_new[label_col].unique())
    labels = [lab for lab in labels_in_test if lab in labels_new]

    if not labels:
        raise RuntimeError("No hay solapamiento de clases entre df_test y df_new_mapped.")

    samples = []

    if mode == "proportional":
        # Distribución actual en test (normalizada)
        dist = (test_counts / test_counts.sum()).loc[labels]

        for label, p in dist.items():
            n_label = int(round(total_new * p))
            pool = df_new[df_new[label_col] == label]

            if len(pool) == 0:
                print(f"[WARN] No hay ejemplos en df_new_mapped para la clase {label}, se salta.")
                continue

            n_label = min(n_label, len(pool))
            if n_label <= 0:
                continue

            samples.append(pool.sample(n=n_label, random_state=random_state))

    elif mode == "balanced":
        # Reparto aproximadamente uniforme entre las clases posibles
        n_classes = len(labels)
        if n_classes == 0:
            raise RuntimeError("No hay clases válidas para modo 'balanced'.")

        base = total_new // n_classes
        remainder = total_new % n_classes

        for idx, label in enumerate(labels):
            pool = df_new[df_new[label_col] == label]

            if len(pool) == 0:
                print(f"[WARN] No hay ejemplos en df_new_mapped para la clase {label}, se salta.")
                continue

            # reparto uniforme base + 1 para las primeras 'remainder' clases
            n_label = base + (1 if idx < remainder else 0)
            n_label = min(n_label, len(pool))

            if n_label <= 0:
                continue

            samples.append(pool.sample(n=n_label, random_state=random_state))

    else:
        raise ValueError("mode debe ser 'proportional' o 'balanced'.")

    if not samples:
        raise RuntimeError("No se han podido muestrear ejemplos nuevos; revisa el solapamiento de clases.")

    df_added = pd.concat(samples, ignore_index=True)

    # Concatenamos al test
    df_test_aug = pd.concat([df_test, df_added], ignore_index=True)

    return df_test_aug, df_added


In [21]:
my_train_df = load_and_clean_tweethate(INPUT_JSONL, text_field="text_masked")
my_test_df = load_and_clean_tweethate("supertweeteval/test_split_only_originals.jsonl", text_field="text_masked")

my_train_df = remove_values_from_column(my_train_df, "predicted_hate_category", ["hate_age"])
my_test_df = remove_values_from_column(my_test_df, "predicted_hate_category", ["hate_age"])

my_train_df = collapse_race_into_origin(my_train_df, col="predicted_hate_category")
my_test_df      = collapse_race_into_origin(my_test_df,      col="predicted_hate_category")

df_tweethate = collapse_race_into_origin(my_train_df, "predicted_hate_category")
df_test      = collapse_race_into_origin(my_test_df,      "predicted_hate_category")

# 2) Mapear dataset nuevo a categorías hate_*
df_new_mapped = map_new_csv_to_hate_categories(df_new_full, text_col="tweet")

# 3) Aumentar el test con, por ejemplo, 300 nuevos ejemplos
df_test_aug, df_added = augment_test_with_new_data(
    df_test, df_new_mapped, total_new=500, mode="balanced"
)

print("Tamaño original test:", len(df_test))
print("Nuevos ejemplos añadidos:", len(df_added))
print("Nuevo tamaño test:", len(df_test_aug))

print("\nDistribución original test:")
print(df_test["predicted_hate_category"].value_counts(normalize=True))

print("\nDistribución test aumentada:")
print(df_test_aug["predicted_hate_category"].value_counts(normalize=True))


Tamaño original test: 104
Nuevos ejemplos añadidos: 468
Nuevo tamaño test: 572

Distribución original test:
predicted_hate_category
hate_gender        0.394231
hate_origin        0.355769
hate_sexuality     0.125000
hate_religion      0.115385
hate_disability    0.009615
Name: proportion, dtype: float64

Distribución test aumentada:
predicted_hate_category
hate_gender        0.246503
hate_origin        0.239510
hate_sexuality     0.197552
hate_disability    0.176573
hate_religion      0.139860
Name: proportion, dtype: float64


In [26]:
INPUT_JSONL = "supertweeteval/merged_masked_unionv2.jsonl"

#INPUT_JSONL = "supertweeteval/broken_phone/balanced_with_broken_phone_supertweetevalv3.jsonl"

my_train_df = load_and_clean_tweethate(INPUT_JSONL, text_field="text_masked")

my_train_df = remove_values_from_column(my_train_df, "predicted_hate_category", ["hate_age"])

my_train_df = collapse_race_into_origin(my_train_df, col="predicted_hate_category")

In [27]:
df_all_runs, df_summary = experiment_from_df_multi_seed(
    df_trainval=my_train_df,
    df_test=df_test_aug,
    X_field="text",
    seeds=[7,11,13,17,19]
)

 Ejecutando Seeds: [7, 11, 13, 17, 19]

##### SEED = 7 #####


=== VAL ===
              precision    recall  f1-score   support

           0     0.1000    0.5000    0.1667         2
           1     0.8966    0.6753    0.7704        77
           2     0.8750    0.6901    0.7717        71
           3     0.5238    0.9565    0.6769        23
           4     0.5938    0.7600    0.6667        25

    accuracy                         0.7222       198
   macro avg     0.5978    0.7164    0.6105       198
weighted avg     0.7992    0.7222    0.7408       198

Confusion matrix:
 [[ 1  0  0  1  0]
 [ 3 52  7  5 10]
 [ 4  4 49 11  3]
 [ 0  1  0 22  0]
 [ 2  1  0  3 19]]
F1-macro: 0.6105
Accuracy: 0.7222

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     0.6081    0.4455    0.5143       101
           1     0.7471    0.4610    0.5702       141
           2     0.6907    0.4891    0.5726       137
           3     0.3067    0.5750    0.4000        80


/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1


=== VAL ===
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.8415    0.8961    0.8679        77
           2     0.8429    0.8310    0.8369        71
           3     0.7500    0.7826    0.7660        23
           4     0.9048    0.7600    0.8261        25

    accuracy                         0.8333       198
   macro avg     0.6678    0.6539    0.6594       198
weighted avg     0.8308    0.8333    0.8309       198

Confusion matrix:
 [[ 0  1  1  0  0]
 [ 1 69  4  1  2]
 [ 0  8 59  4  0]
 [ 0  2  3 18  0]
 [ 0  2  3  1 19]]
F1-macro: 0.6594
Accuracy: 0.8333

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     0.7857    0.3267    0.4615       101
           1     0.5667    0.6028    0.5842       141
           2     0.5185    0.7153    0.6012       137
           3     0.6122    0.3750    0.4651        80
           4     0.7042    0.8850    0.7843       113

    acc

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.9315    0.8831    0.9067        77
           2     0.9143    0.9014    0.9078        71
           3     0.7241    0.9130    0.8077        23
           4     0.9200    0.9200    0.9200        25

    accuracy                         0.8889       198
   macro avg     0.6980    0.7235    0.7084       198
weighted avg     0.8904    0.8889    0.8881       198

Confusion matrix:
 [[ 0  1  0  0  1]
 [ 1 68  5  2  1]
 [ 0  3 64  4  0]
 [ 0  1  1 21  0]
 [ 0  0  0  2 23]]
F1-macro: 0.7084
Accuracy: 0.8889

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     0.7000    0.4158    0.5217       101
           1     0.6783    0.5532    0.6094       141
           2     0.5641    0.6423    0.6007       137
           3     0.4819    0.5000    0.4908        80
           4     0.6456    0.9027    0.7528       113

    accuracy        

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(


In [28]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.704571,0.045734,0.664483,0.744658,0.635265,0.034575,0.604959,0.665571
1,linear_svm,0.691243,0.049450,0.647898,0.734588,0.626857,0.043943,0.588339,0.665375
3,passive_aggressive,0.689503,0.058110,0.638567,0.740439,0.614215,0.037882,0.581010,0.647420
2,logreg_ovr,0.621616,0.023852,0.600708,0.642524,0.587858,0.036573,0.555800,0.619916
0,complement_nb,0.584789,0.029115,0.559268,0.610309,0.570786,0.029560,0.544875,0.596697


In [25]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
3,passive_aggressive,0.986146,0.004920,0.981834,0.990459,0.573849,0.011785,0.563520,0.584179
1,linear_svm,0.982166,0.003895,0.978752,0.985580,0.625455,0.006698,0.619584,0.631326
4,sgd_log,0.975202,0.004870,0.970934,0.979471,0.627862,0.006536,0.622132,0.633591
2,logreg_ovr,0.964467,0.006119,0.959103,0.969830,0.638500,0.001303,0.637358,0.639642
0,complement_nb,0.931495,0.014556,0.918736,0.944254,0.626433,0.004067,0.622868,0.629998


## Remasking new test and test again

In [29]:
def load_lexicon_from_jsonl(path="lexicon/lexicon.jsonl"):
    """
    Lee lexicon/lexicon.jsonl y devuelve un dict Python con el lexicon.
    Asume que la primera línea contiene el lexicon (o un campo 'lexicon').
    """
    df_lex = load_jsonl(path)
    if len(df_lex) == 0:
        raise RuntimeError(f"No se encontró ninguna línea en {path}")

    row0 = df_lex.iloc[0]

    # Caso 1: columna explícita llamada 'lexicon'
    if "lexicon" in df_lex.columns and isinstance(row0["lexicon"], dict):
        return row0["lexicon"]

    # Caso 2: solo una columna y esa columna contiene el dict
    if len(df_lex.columns) == 1:
        col = df_lex.columns[0]
        if isinstance(row0[col], dict):
            return row0[col]

    # Fallback: convertir la fila entera a dict
    maybe_dict = row0.to_dict()
    if isinstance(maybe_dict, dict):
        return maybe_dict

    raise RuntimeError("Formato de lexicon.jsonl no reconocido.")


In [30]:
import re
import numpy as np
import pandas as pd

def build_lexicon_entries(lexicon, placeholder_keys=("TARGET", "SLUR")):
    """
    Flatten the lexicon into a list of entries:

        {
            "phrase": original phrase (str),
            "pattern": compiled regex,
            "placeholder": "[TARGET:GENDER_WOMEN]",
            "group": "GENDER_WOMEN",
            "kind": "TARGET" or "SLUR",
        }

    - Ignores *very* short phrases (len < 2) to avoid noise like "u".
    - For “normal word” phrases (letters/digits/spaces/'/-), we add word
      boundaries, so we match complete words, not substrings inside words.
    """
    entries = []

    for kind in placeholder_keys:  # "TARGET", "SLUR"
        if kind not in lexicon:
            continue

        by_axis = lexicon[kind]   # e.g. gender / ethnicity / ...
        for axis, by_group in by_axis.items():
            for group_code, phrase_dict in by_group.items():
                for phrase in phrase_dict.keys():
                    if not phrase:
                        continue

                    phrase_stripped = phrase.strip()
                    if len(phrase_stripped) < 2:
                        # skip ultra-short stuff: "u", etc.
                        continue

                    # "Normal word" (letters/digits/space/'/-) → use word boundaries
                    if re.fullmatch(r"[0-9A-Za-z' \-]+", phrase_stripped):
                        pattern = re.compile(
                            r"\b" + re.escape(phrase_stripped) + r"\b",
                            flags=re.IGNORECASE,
                        )
                    else:
                        # weird chars / emoji / punctuation → plain literal match
                        pattern = re.compile(re.escape(phrase_stripped), flags=re.IGNORECASE)

                    entries.append({
                        "phrase": phrase_stripped,
                        "pattern": pattern,
                        "placeholder": f"[{kind}:{group_code}]",
                        "group": group_code,
                        "kind": kind,
                    })

    # Longer phrases first (reduces conflicts: "black women" before "black")
    entries.sort(key=lambda e: len(e["phrase"]), reverse=True)
    return entries





In [31]:
import re

def mask_text_with_lexicon(text, entries):
    if not isinstance(text, str) or not text:
        return text, []

    n = len(text)
    occupied = [False] * n
    spans = []   # (start, end, placeholder, group)
    groups = set()
    seen_placeholders = set()

    for e in entries:
        pattern = e["pattern"]

        for m in pattern.finditer(text):
            start, end = m.span()
            if start == end:
                continue

            # ya cubierto por un span más largo
            if any(occupied[i] for i in range(start, end)):
                continue

            # NUEVO: solo una vez por placeholder
            if e["placeholder"] in seen_placeholders:
                continue

            spans.append((start, end, e["placeholder"], e["group"]))
            for i in range(start, end):
                occupied[i] = True
            groups.add(e["group"])
            seen_placeholders.add(e["placeholder"])

    if not spans:
        return text, []

    spans.sort(key=lambda x: x[0])

    out = []
    last = 0
    for start, end, placeholder, _group in spans:
        out.append(text[last:start])
        out.append(placeholder)
        last = end
    out.append(text[last:])

    masked_text = "".join(out)
    return masked_text, sorted(groups)


In [32]:
def remask_df_keep_minimal(df, lexicon_path="lexicon/lexicon.jsonl"):
    cols_needed = ["text", "text_masked", "predicted_hate_category"]
    missing = [c for c in cols_needed if c not in df.columns]
    if missing:
        raise KeyError(f"Faltan columnas en df: {missing}")

    df_out = df[cols_needed].copy()

    lexicon = load_lexicon_from_jsonl(lexicon_path)
    entries = build_lexicon_entries(lexicon)

    mask_missing = df_out["text_masked"].isna() | df_out["text_masked"].astype(str).str.strip().eq("")
    n_to_remask = mask_missing.sum()
    print(f"Remasking {n_to_remask} filas con text_masked vacío/NaN...")

    for idx in df_out[mask_missing].index:
        original_text = df_out.at[idx, "text"]
        masked_text, groups = mask_text_with_lexicon(original_text, entries)

        if isinstance(masked_text, str):
            df_out.at[idx, "text_masked"] = f'"""{masked_text}"""'
        else:
            df_out.at[idx, "text_masked"] = masked_text

    return df_out


In [33]:
# df_merged es tu dataframe grande con muchas columnas
df_min_remasked = remask_df_keep_minimal(
    df_test_aug,
    lexicon_path="lexicon/lexicon.jsonl"
)

print(df_min_remasked.head())
print(df_min_remasked.columns)  # solo text, text_masked, predicted_hate_category


Remasking 468 filas con text_masked vacío/NaN...
                                                text  \
0                                   Fck those niggas   
1  @user . Linda Sarsour should already be in jai...   
2  Women really should start carrying guns around...   
3  @mike_pence @user Just shut your mouth and put...   
4  Please let that bitch go. Every man done ran i...   

                                         text_masked predicted_hate_category  
0                  """Fck those [SLUR:RACE_BLACK]"""             hate_origin  
1  """[MENTION] . Linda Sarsour should already be...             hate_gender  
2  """[GENDER_WOMEN] really should start carrying...             hate_gender  
3  """[MENTION] [MENTION] Just shut your mouth an...          hate_sexuality  
4  """Please let that [SLUR:GENDER_WOMEN] go. Eve...             hate_gender  
Index(['text', 'text_masked', 'predicted_hate_category'], dtype='object')


In [34]:
df_min_remasked

,text,text_masked,predicted_hate_category
0,Fck those niggas,"""""""Fck those [SLUR:RACE_BLACK]""""""",hate_origin
1,@user . Linda Sarsour should already be in jai...,"""""""[MENTION] . Linda Sarsour should already be...",hate_gender
2,Women really should start carrying guns around...,"""""""[GENDER_WOMEN] really should start carrying...",hate_gender
3,@mike_pence @user Just shut your mouth and put...,"""""""[MENTION] [MENTION] Just shut your mouth an...",hate_sexuality
4,Please let that bitch go. Every man done ran i...,"""""""Please let that [SLUR:GENDER_WOMEN] go. Eve...",hate_gender
...,...,...,...
567,@user @user @user @user @user @user Saw this t...,"""""""[SLUR:GENDER_MEN] [SLUR:GENDER_WOMEN] [SLUR...",hate_disability
568,okay but really why are Americans so fucking m...,"""""""okay but really why are [TARGET:NATIONAL_OR...",hate_disability
569,@user what\u2019s wrong with you you fucking r...,"""""""[SLUR:GENDER_MEN] what\u2019s wrong with yo...",hate_disability
570,@user @user @user @user Hell I'd say enticed t...,"""""""[SLUR:GENDER_MEN] [SLUR:GENDER_WOMEN] [SLUR...",hate_disability


In [57]:
INPUT_JSONL = "supertweeteval/merged_masked_unionv2.jsonl"

#INPUT_JSONL = "supertweeteval/broken_phone/balanced_with_broken_phone_supertweetevalv3.jsonl"

my_train_df = load_and_clean_tweethate(INPUT_JSONL, text_field="text_masked")

my_train_df = remove_values_from_column(my_train_df, "predicted_hate_category", ["hate_age"])

my_train_df = collapse_race_into_origin(my_train_df, col="predicted_hate_category")

In [58]:
df_all_runs, df_summary = experiment_from_df_multi_seed(
    df_trainval=my_train_df,
    df_test=df_min_remasked,
    X_field="text",
    seeds=[7,11,13,17,19]
)

 Ejecutando Seeds: [7, 11, 13, 17, 19]

##### SEED = 7 #####


=== VAL ===
              precision    recall  f1-score   support

           0     0.1000    0.5000    0.1667         2
           1     0.8966    0.6753    0.7704        77
           2     0.8750    0.6901    0.7717        71
           3     0.5238    0.9565    0.6769        23
           4     0.5938    0.7600    0.6667        25

    accuracy                         0.7222       198
   macro avg     0.5978    0.7164    0.6105       198
weighted avg     0.7992    0.7222    0.7408       198

Confusion matrix:
 [[ 1  0  0  1  0]
 [ 3 52  7  5 10]
 [ 4  4 49 11  3]
 [ 0  1  0 22  0]
 [ 2  1  0  3 19]]
F1-macro: 0.6105
Accuracy: 0.7222

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     0.6081    0.4455    0.5143       101
           1     0.7471    0.4610    0.5702       141
           2     0.6907    0.4891    0.5726       137
           3     0.3067    0.5750    0.4000        80


/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1


=== VAL ===
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.8415    0.8961    0.8679        77
           2     0.8429    0.8310    0.8369        71
           3     0.7500    0.7826    0.7660        23
           4     0.9048    0.7600    0.8261        25

    accuracy                         0.8333       198
   macro avg     0.6678    0.6539    0.6594       198
weighted avg     0.8308    0.8333    0.8309       198

Confusion matrix:
 [[ 0  1  1  0  0]
 [ 1 69  4  1  2]
 [ 0  8 59  4  0]
 [ 0  2  3 18  0]
 [ 0  2  3  1 19]]
F1-macro: 0.6594
Accuracy: 0.8333

=== TEST (fijo) ===
              precision    recall  f1-score   support

           0     0.7857    0.3267    0.4615       101
           1     0.5667    0.6028    0.5842       141
           2     0.5185    0.7153    0.6012       137
           3     0.6122    0.3750    0.4651        80
           4     0.7042    0.8850    0.7843       113

    acc

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1

              precision    recall  f1-score   support

           0     0.7000    0.4158    0.5217       101
           1     0.6783    0.5532    0.6094       141
           2     0.5641    0.6423    0.6007       137
           3     0.4819    0.5000    0.4908        80
           4     0.6456    0.9027    0.7528       113

    accuracy                         0.6119       572
   macro avg     0.6140    0.6028    0.5951       572
weighted avg     0.6208    0.6119    0.6036       572

Confusion matrix:
 [[ 42  11  24  12  12]
 [  7  78  24  12  20]
 [  6   7  88  17  19]
 [  3  14  18  40   5]
 [  2   5   2   2 102]]
F1-macro: 0.5951
Accuracy: 0.6119

=== VAL ===
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.9200    0.8961    0.9079        77
           2     0.8904    0.9155    0.9028        71
           3     0.7857    0.9565    0.8627        23
           4     1.0000    0.8400    0.9130        25

   

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(


In [59]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.704571,0.045734,0.664483,0.744658,0.635265,0.034575,0.604959,0.665571
1,linear_svm,0.691243,0.049450,0.647898,0.734588,0.626857,0.043943,0.588339,0.665375
3,passive_aggressive,0.689503,0.058110,0.638567,0.740439,0.614215,0.037882,0.581010,0.647420
2,logreg_ovr,0.621616,0.023852,0.600708,0.642524,0.587858,0.036573,0.555800,0.619916
0,complement_nb,0.584789,0.029115,0.559268,0.610309,0.570786,0.029560,0.544875,0.596697


In [56]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
3,passive_aggressive,0.986146,0.004920,0.981834,0.990459,0.573849,0.011785,0.563520,0.584179
1,linear_svm,0.982166,0.003895,0.978752,0.985580,0.625455,0.006698,0.619584,0.631326
4,sgd_log,0.975202,0.004870,0.970934,0.979471,0.627862,0.006536,0.622132,0.633591
2,logreg_ovr,0.964467,0.006119,0.959103,0.969830,0.638500,0.001303,0.637358,0.639642
0,complement_nb,0.931495,0.014556,0.918736,0.944254,0.626433,0.004067,0.622868,0.629998


# Dual view

In [41]:
from scipy.sparse import hstack

def _single_seed_experiment_dualview(
    df_trainval,
    df_test,
    raw_field,
    masked_field,
    seed,
    val_size,
    ngram_range,
    min_df,
    max_df,
    max_features
):
    """
    Single-seed experiment using BOTH raw and masked text as features.

    df_trainval / df_test must contain:
        - raw_field        (e.g. 'text')
        - masked_field     (e.g. 'text_masked')
        - predicted_hate_category
    """
    random.seed(seed)
    np.random.seed(seed)

    # --------- Filter valid text in both views ----------
    def valid_str(x):
        return isinstance(x, str) and x.strip() != ""

    df_trainval = df_trainval[
        df_trainval[raw_field].apply(valid_str)
        & df_trainval[masked_field].apply(valid_str)
    ].copy()

    df_test = df_test[
        df_test[raw_field].apply(valid_str)
        & df_test[masked_field].apply(valid_str)
    ].copy()

    # --------- Normalize and filter 'nan' labels ----------
    for df in (df_trainval, df_test):
        df["phc"] = df["predicted_hate_category"].astype(str).str.strip()

    df_trainval = df_trainval[df_trainval["phc"].str.lower() != "nan"].copy()
    df_test     = df_test[df_test["phc"].str.lower() != "nan"].copy()

    if len(df_trainval) == 0 or len(df_test) == 0:
        raise RuntimeError("Train/Val o Test vacío tras filtrar 'nan'.")

    # --------- Label mapping (union train+test) ----------
    all_labels = sorted(set(df_trainval["phc"].str.lower()) |
                        set(df_test["phc"].str.lower()))
    label2id = {c: i for i, c in enumerate(all_labels)}
    id2label = {i: c for c, i in label2id.items()}

    for df in (df_trainval, df_test):
        df["label"] = df["phc"].str.lower().map(label2id).astype(int)
        df["clean_raw"]    = df[raw_field].apply(clean_text)
        df["clean_masked"] = df[masked_field].apply(clean_text)

    # --------- Train/Val split (dual-view) ----------
    X_raw = df_trainval["clean_raw"].values
    X_msk = df_trainval["clean_masked"].values
    y_all = df_trainval["label"].values

    X_raw_train, X_raw_val, X_msk_train, X_msk_val, y_train, y_val = train_test_split(
        X_raw,
        X_msk,
        y_all,
        test_size=val_size,
        random_state=seed,
        stratify=y_all,
    )

    X_raw_test = df_test["clean_raw"].values
    X_msk_test = df_test["clean_masked"].values
    y_test     = df_test["label"].values

    # --------- TF-IDF for RAW and MASKED separately ----------
    vec_raw = TfidfVectorizer(
        lowercase=True,
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        strip_accents="unicode",
        sublinear_tf=True,
        stop_words="english",
        max_features=max_features,
    )
    vec_msk = TfidfVectorizer(
        lowercase=True,
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        strip_accents="unicode",
        sublinear_tf=True,
        stop_words="english",
        max_features=max_features,
    )

    Xtr_raw = vec_raw.fit_transform(X_raw_train)
    Xva_raw = vec_raw.transform(X_raw_val)
    Xte_raw = vec_raw.transform(X_raw_test)

    Xtr_msk = vec_msk.fit_transform(X_msk_train)
    Xva_msk = vec_msk.transform(X_msk_val)
    Xte_msk = vec_msk.transform(X_msk_test)

    # Concatenate [raw | masked]
    Xtr = hstack([Xtr_raw, Xtr_msk], format="csr")
    Xva = hstack([Xva_raw, Xva_msk], format="csr")
    Xte = hstack([Xte_raw, Xte_msk], format="csr")

    # --------- Class weights ----------
    uniq = np.unique(y_train)
    class_weights = compute_class_weight(class_weight="balanced", classes=uniq, y=y_train)
    cw_map = {int(c): float(w) for c, w in zip(uniq, class_weights)}
    sw_train = np.vectorize(cw_map.get)(y_train)

    # --------- Models ----------
    models = {
        "logreg_ovr": LogisticRegression(
            max_iter=8000, n_jobs=-1, class_weight=cw_map, C=2.0,
            solver="liblinear", multi_class="ovr", random_state=seed),
        "linear_svm": LinearSVC(class_weight=cw_map, random_state=seed),
        "sgd_log": SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=2000,
                                 tol=1e-3, random_state=seed),
        "passive_aggressive": PassiveAggressiveClassifier(C=0.5, max_iter=2000,
                                                          random_state=seed),
        "complement_nb": ComplementNB(alpha=0.5),
    }

    seed_results = []

    for name, clf in models.items():
        print("\n" + "="*100)
        print(f"Entrenando (dual-view): {name}  — seed={seed}")

        # Entrenamiento
        try:
            clf.fit(Xtr, y_train, sample_weight=sw_train)
        except TypeError:
            clf.fit(Xtr, y_train)

        # Evaluación
        val_metrics  = eval_split(clf, Xva, y_val, "VAL (dual-view)")
        test_metrics = eval_split(clf, Xte, y_test, "TEST (dual-view, fijo)")

        seed_results.append({
            "model": name,
            "seed": seed,
            "val_f1_macro":  val_metrics["f1_macro"],
            "val_accuracy":  val_metrics["accuracy"],
            "test_f1_macro": test_metrics["f1_macro"],
            "test_accuracy": test_metrics["accuracy"],
        })

    return pd.DataFrame(seed_results)


In [ ]:
def experiment_dualview_from_df_multi_seed(
        df_trainval,
        df_test,
        raw_field="text",
        masked_field="text_masked",
        seeds=[1,2,3,4,5],
        val_size=0.10,
        ngram_range=(1,3),
        min_df=3,
        max_df=0.90,
        max_features=None):

    all_runs = []

    print("=====================================")
    print(" Ejecutando Seeds (dual-view):", seeds)
    print(" raw_field   :", raw_field)
    print(" masked_field:", masked_field)
    print("=====================================")

    for sd in seeds:
        print(f"\n##### SEED = {sd} #####\n")
        df_seed = _single_seed_experiment_dualview(
            df_trainval.copy(),
            df_test.copy(),
            raw_field=raw_field,
            masked_field=masked_field,
            seed=sd,
            val_size=val_size,
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            max_features=max_features
        )
        all_runs.append(df_seed)

    df_all = pd.concat(all_runs, ignore_index=True)

    # ============================
    #  Aggregate by model (95% CI)
    # ============================
    rows = []
    for model, sub in df_all.groupby("model"):
        mean_val, std_val, lo_val, hi_val = ci95(sub["val_f1_macro"])
        mean_te, std_te, lo_te, hi_te = ci95(sub["test_f1_macro"])
I
        rows.append({
            "model": model,
            "val_f1_macro_mean":    mean_val,
            "val_f1_macro_std":     std_val,
            "val_f1_macro_CI_low":  lo_val,
            "val_f1_macro_CI_high": hi_val,

            "test_f1_macro_mean":    mean_te,
            "test_f1_macro_std":     std_te,
            "test_f1_macro_CI_low":  lo_te,
            "test_f1_macro_CI_high": hi_te,
        })

    df_summary = pd.DataFrame(rows).sort_values(
        by="val_f1_macro_mean", ascending=False
    )

    print("\n=====================================")
    print("Resumen Multi-Seed (dual-view, 95% CI)")
    print("=====================================")
    print(df_summary.to_string(index=False))

    return df_all, df_summary


In [47]:
INPUT_JSONL = "supertweeteval/merged_masked_unionv2.jsonl"
TEST_JSONL = "supertweeteval/test_split_only_originals.jsonl"

#INPUT_JSONL = "supertweeteval/broken_phone/balanced_with_broken_phone_supertweetevalv3.jsonl"

my_train_df = load_and_clean_tweethate(INPUT_JSONL, text_field="text_masked")
my_train_df = remove_values_from_column(my_train_df, "predicted_hate_category", ["hate_age"])
my_train_df = collapse_race_into_origin(my_train_df, col="predicted_hate_category")

my_test_df = load_and_clean_tweethate(TEST_JSONL, text_field="text_masked")
my_test_df = remove_values_from_column(my_test_df, "predicted_hate_category", ["hate_age"])
my_test_df = collapse_race_into_origin(my_test_df, col="predicted_hate_category")

df_all_dual, df_summary_dual = experiment_dualview_from_df_multi_seed(
    df_trainval=my_train_df,
    df_test=my_test_df,
    raw_field="text",          # o el campo que uses para raw
    masked_field="text_masked",
    seeds=[7, 11, 13, 17, 19],
)

df_summary_dual

 Ejecutando Seeds (dual-view): [7, 11, 13, 17, 19]
 raw_field   : text
 masked_field: text_masked

##### SEED = 7 #####


Entrenando (dual-view): logreg_ovr  — seed=7

=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     0.3333    1.0000    0.5000         2
           1     0.9403    0.8182    0.8750        77
           2     0.9104    0.8592    0.8841        71
           3     0.7500    0.9130    0.8235        23
           4     0.8000    0.9600    0.8727        25

    accuracy                         0.8636       198
   macro avg     0.7468    0.9101    0.7911       198
weighted avg     0.8836    0.8636    0.8682       198

Confusion matrix:
 [[ 2  0  0  0  0]
 [ 2 63  6  2  4]
 [ 2  2 61  5  1]
 [ 0  1  0 21  1]
 [ 0  1  0  0 24]]
F1-macro: 0.7911
Accuracy: 0.8636

=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.3333    1.0000    0.5000         1
           1     0.8750    0.8537 

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1


=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.3333    1.0000    0.5000         1
           1     0.9211    0.8537    0.8861        41
           2     0.9643    0.7297    0.8308        37
           3     0.5000    0.7500    0.6000        12
           4     0.7647    1.0000    0.8667        13

    accuracy                         0.8173       104
   macro avg     0.6967    0.8667    0.7367       104
weighted avg     0.8627    0.8173    0.8273       104

Confusion matrix:
 [[ 1  0  0  0  0]
 [ 0 35  1  3  2]
 [ 2  2 27  6  0]
 [ 0  1  0  9  2]
 [ 0  0  0  0 13]]
F1-macro: 0.7367
Accuracy: 0.8173

##### SEED = 11 #####


Entrenando (dual-view): logreg_ovr  — seed=11

=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     0.2500    0.5000    0.3333         2
           1     0.9706    0.8571    0.9103        77
           2     0.9841    0.8732    0.9254        71
           3     0.718

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.1


=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     0.5000    0.5000    0.5000         2
           1     0.9231    0.9351    0.9290        77
           2     0.9275    0.9014    0.9143        71
           3     0.8214    1.0000    0.9020        23
           4     1.0000    0.8400    0.9130        25

    accuracy                         0.9141       198
   macro avg     0.8344    0.8353    0.8317       198
weighted avg     0.9183    0.9141    0.9142       198

Confusion matrix:
 [[ 1  0  1  0  0]
 [ 1 72  3  1  0]
 [ 0  4 64  3  0]
 [ 0  0  0 23  0]
 [ 0  2  1  1 21]]
F1-macro: 0.8317
Accuracy: 0.9141

=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.5000    1.0000    0.6667         1
           1     0.8810    0.9024    0.8916        41
           2     0.8750    0.9459    0.9091        37
           3     0.8571    0.5000    0.6316        12
           4     0.8462    0.8462    0.

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(


,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.812553,0.047141,0.771232,0.853874,0.777320,0.016304,0.763028,0.791611
1,linear_svm,0.806271,0.031544,0.778621,0.833921,0.782299,0.012419,0.771413,0.793185
3,passive_aggressive,0.797912,0.044321,0.759063,0.836761,0.782654,0.015031,0.769479,0.795829
2,logreg_ovr,0.787567,0.037245,0.754920,0.820214,0.729400,0.018422,0.713253,0.745548
0,complement_nb,0.698375,0.022067,0.679032,0.717718,0.712254,0.020343,0.694422,0.730086


In [48]:
df_summary_dual

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.812553,0.047141,0.771232,0.853874,0.777320,0.016304,0.763028,0.791611
1,linear_svm,0.806271,0.031544,0.778621,0.833921,0.782299,0.012419,0.771413,0.793185
3,passive_aggressive,0.797912,0.044321,0.759063,0.836761,0.782654,0.015031,0.769479,0.795829
2,logreg_ovr,0.787567,0.037245,0.754920,0.820214,0.729400,0.018422,0.713253,0.745548
0,complement_nb,0.698375,0.022067,0.679032,0.717718,0.712254,0.020343,0.694422,0.730086


In [46]:
df_summary_dual

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
1,linear_svm,0.986012,0.005964,0.980784,0.991240,0.876819,0.036833,0.844533,0.909104
3,passive_aggressive,0.985681,0.008635,0.978112,0.993249,0.882591,0.013909,0.870399,0.894782
4,sgd_log,0.980131,0.008463,0.972712,0.987549,0.827095,0.002755,0.824680,0.829510
2,logreg_ovr,0.973096,0.009082,0.965135,0.981056,0.812155,0.004307,0.808379,0.815930
0,complement_nb,0.948422,0.008715,0.940782,0.956061,0.794841,0.015081,0.781622,0.808060


In [49]:
df_min_remasked

,text,text_masked,predicted_hate_category
0,Fck those niggas,"""""""Fck those [SLUR:RACE_BLACK]""""""",hate_origin
1,@user . Linda Sarsour should already be in jai...,"""""""[MENTION] . Linda Sarsour should already be...",hate_gender
2,Women really should start carrying guns around...,"""""""[GENDER_WOMEN] really should start carrying...",hate_gender
3,@mike_pence @user Just shut your mouth and put...,"""""""[MENTION] [MENTION] Just shut your mouth an...",hate_sexuality
4,Please let that bitch go. Every man done ran i...,"""""""Please let that [SLUR:GENDER_WOMEN] go. Eve...",hate_gender
...,...,...,...
567,@user @user @user @user @user @user Saw this t...,"""""""[SLUR:GENDER_MEN] [SLUR:GENDER_WOMEN] [SLUR...",hate_disability
568,okay but really why are Americans so fucking m...,"""""""okay but really why are [TARGET:NATIONAL_OR...",hate_disability
569,@user what\u2019s wrong with you you fucking r...,"""""""[SLUR:GENDER_MEN] what\u2019s wrong with yo...",hate_disability
570,@user @user @user @user Hell I'd say enticed t...,"""""""[SLUR:GENDER_MEN] [SLUR:GENDER_WOMEN] [SLUR...",hate_disability


In [63]:
#INPUT_JSONL = "supertweeteval/merged_masked_unionv2.jsonl"
TEST_JSONL = "supertweeteval/test_split_only_originals.jsonl"

INPUT_JSONL = "supertweeteval/broken_phone/balanced_with_broken_phone_supertweetevalv3.jsonl"

my_train_df = load_and_clean_tweethate(INPUT_JSONL, text_field="text_masked")
my_train_df = remove_values_from_column(my_train_df, "predicted_hate_category", ["hate_age"])
my_train_df = collapse_race_into_origin(my_train_df, col="predicted_hate_category")

my_test_df = load_and_clean_tweethate(TEST_JSONL, text_field="text_masked")
my_test_df = remove_values_from_column(my_test_df, "predicted_hate_category", ["hate_age"])
my_test_df = collapse_race_into_origin(my_test_df, col="predicted_hate_category")

df_all_dual, df_summary_dual = experiment_dualview_from_df_multi_seed(
    df_trainval=my_train_df,
    df_test=df_min_remasked,
    raw_field="text",          # o el campo que uses para raw
    masked_field="text_masked",
    seeds=[7, 11, 13, 17, 19],
)

df_summary_dual

 Ejecutando Seeds (dual-view): [7, 11, 13, 17, 19]
 raw_field   : text
 masked_field: text_masked

##### SEED = 7 #####


Entrenando (dual-view): logreg_ovr  — seed=7

=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        71
           1     0.9359    0.9733    0.9542        75
           2     1.0000    0.9730    0.9863       148
           3     1.0000    1.0000    1.0000        74
           4     0.9737    0.9867    0.9801        75

    accuracy                         0.9842       443
   macro avg     0.9819    0.9866    0.9841       443
weighted avg     0.9847    0.9842    0.9843       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  0  73   0   0   2]
 [  0   4 144   0   0]
 [  0   0   0  74   0]
 [  0   1   0   0  74]]
F1-macro: 0.9841
Accuracy: 0.9842

=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.8667    0.3861    0.5342       101
         

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(



Entrenando (dual-view): logreg_ovr  — seed=11

=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        71
           1     0.8452    0.9467    0.8931        75
           2     0.9858    0.9392    0.9619       148
           3     0.9863    0.9730    0.9796        74
           4     0.9730    0.9600    0.9664        75

    accuracy                         0.9594       443
   macro avg     0.9581    0.9638    0.9602       443
weighted avg     0.9622    0.9594    0.9601       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  0  71   2   0   2]
 [  0   8 139   1   0]
 [  0   2   0  72   0]
 [  0   3   0   0  72]]
F1-macro: 0.9602
Accuracy: 0.9594

=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.8636    0.3762    0.5241       101
           1     0.4851    0.8085    0.6064       141
           2     0.7391    0.4964    0.5939       137
           3     0.63

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(



Entrenando (dual-view): logreg_ovr  — seed=13

=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     0.9861    1.0000    0.9930        71
           1     0.9114    0.9600    0.9351        75
           2     0.9861    0.9595    0.9726       148
           3     1.0000    1.0000    1.0000        74
           4     1.0000    0.9867    0.9933        75

    accuracy                         0.9774       443
   macro avg     0.9767    0.9812    0.9788       443
weighted avg     0.9781    0.9774    0.9776       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  1  72   2   0   0]
 [  0   6 142   0   0]
 [  0   0   0  74   0]
 [  0   1   0   0  74]]
F1-macro: 0.9788
Accuracy: 0.9774

=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.8696    0.3960    0.5442       101
           1     0.4936    0.8156    0.6150       141
           2     0.7263    0.5036    0.5948       137
           3     0.61

...


Entrenando (dual-view): logreg_ovr  — seed=17

=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        71
           1     0.8987    0.9467    0.9221        75
           2     0.9859    0.9459    0.9655       148
           3     0.9867    1.0000    0.9933        74
           4     0.9737    0.9867    0.9801        75

    accuracy                         0.9707       443
   macro avg     0.9690    0.9759    0.9722       443
weighted avg     0.9715    0.9707    0.9708       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  0  71   2   1   1]
 [  0   7 140   0   1]
 [  0   0   0  74   0]
 [  0   1   0   0  74]]
F1-macro: 0.9722
Accuracy: 0.9707

=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.8636    0.3762    0.5241       101
           1     0.4893    0.8085    0.6096       141
           2     0.6837    0.4891    0.5702       137
           3     0.63

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(



Entrenando (dual-view): logreg_ovr  — seed=19

=== VAL (dual-view) ===
              precision    recall  f1-score   support

           0     0.9861    1.0000    0.9930        71
           1     0.9577    0.9067    0.9315        75
           2     0.9658    0.9527    0.9592       148
           3     0.9610    1.0000    0.9801        74
           4     0.9740    1.0000    0.9868        75

    accuracy                         0.9684       443
   macro avg     0.9689    0.9719    0.9701       443
weighted avg     0.9683    0.9684    0.9681       443

Confusion matrix:
 [[ 71   0   0   0   0]
 [  1  68   5   0   1]
 [  0   3 141   3   1]
 [  0   0   0  74   0]
 [  0   0   0   0  75]]
F1-macro: 0.9701
Accuracy: 0.9684

=== TEST (dual-view, fijo) ===
              precision    recall  f1-score   support

           0     0.8723    0.4059    0.5541       101
           1     0.5067    0.8014    0.6209       141
           2     0.7100    0.5182    0.5992       137
           3     0.63

/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/mrodrive/miniconda3/envs/hs-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(


,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
1,linear_svm,0.986012,0.005964,0.980784,0.991240,0.606916,0.009183,0.598866,0.614965
3,passive_aggressive,0.985681,0.008635,0.978112,0.993249,0.589827,0.010742,0.580411,0.599243
4,sgd_log,0.980131,0.008463,0.972712,0.987549,0.604594,0.006636,0.598778,0.610411
2,logreg_ovr,0.973096,0.009082,0.965135,0.981056,0.595477,0.007028,0.589317,0.601638
0,complement_nb,0.948422,0.008715,0.940782,0.956061,0.654592,0.006824,0.648610,0.660573


In [62]:
df_summary_dual

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.812553,0.047141,0.771232,0.853874,0.640961,0.013899,0.628777,0.653144
1,linear_svm,0.806271,0.031544,0.778621,0.833921,0.636804,0.010158,0.627900,0.645707
3,passive_aggressive,0.797912,0.044321,0.759063,0.836761,0.618875,0.025060,0.596909,0.640841
2,logreg_ovr,0.787567,0.037245,0.754920,0.820214,0.638123,0.012751,0.626947,0.649300
0,complement_nb,0.698375,0.022067,0.679032,0.717718,0.635388,0.010188,0.626458,0.644318


In [64]:
df_summary_dual

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
1,linear_svm,0.986012,0.005964,0.980784,0.991240,0.606916,0.009183,0.598866,0.614965
3,passive_aggressive,0.985681,0.008635,0.978112,0.993249,0.589827,0.010742,0.580411,0.599243
4,sgd_log,0.980131,0.008463,0.972712,0.987549,0.604594,0.006636,0.598778,0.610411
2,logreg_ovr,0.973096,0.009082,0.965135,0.981056,0.595477,0.007028,0.589317,0.601638
0,complement_nb,0.948422,0.008715,0.940782,0.956061,0.654592,0.006824,0.648610,0.660573


In [ ]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
4,sgd_log,0.704571,0.045734,0.664483,0.744658,0.635265,0.034575,0.604959,0.665571
1,linear_svm,0.691243,0.049450,0.647898,0.734588,0.626857,0.043943,0.588339,0.665375
3,passive_aggressive,0.689503,0.058110,0.638567,0.740439,0.614215,0.037882,0.581010,0.647420
2,logreg_ovr,0.621616,0.023852,0.600708,0.642524,0.587858,0.036573,0.555800,0.619916
0,complement_nb,0.584789,0.029115,0.559268,0.610309,0.570786,0.029560,0.544875,0.596697


In [ ]:
df_summary

,model,val_f1_macro_mean,val_f1_macro_std,val_f1_macro_CI_low,val_f1_macro_CI_high,test_f1_macro_mean,test_f1_macro_std,test_f1_macro_CI_low,test_f1_macro_CI_high
3,passive_aggressive,0.986146,0.004920,0.981834,0.990459,0.573849,0.011785,0.563520,0.584179
1,linear_svm,0.982166,0.003895,0.978752,0.985580,0.625455,0.006698,0.619584,0.631326
4,sgd_log,0.975202,0.004870,0.970934,0.979471,0.627862,0.006536,0.622132,0.633591
2,logreg_ovr,0.964467,0.006119,0.959103,0.969830,0.638500,0.001303,0.637358,0.639642
0,complement_nb,0.931495,0.014556,0.918736,0.944254,0.626433,0.004067,0.622868,0.629998
